# Notebook 10 — Lang-PINN: LLM-Guided PINNs

**What you'll learn:**
- What Lang-PINN is and why it matters
- The 3 operating modes: library, code-agent, hybrid
- How each agent works: PDE Agent, PINN Agent, Code Agent
- Running the full pipeline from natural language to generated code

**Prerequisites:** Notebooks 01-08 (PINN fundamentals). No LLM experience needed.

**Time:** ~30 minutes

## The Problem

By now you've seen how PINNs work — define a PDE, write loss functions, train a network, evaluate. The core library (`pinn`) handles the training mechanics, but you still need to:

1. **Know the math** — write the PDE residual in code
2. **Choose architecture** — how many layers, neurons, which activation, Ansatz?
3. **Set hyperparameters** — learning rate, loss weights, collocation density

What if you could just **describe the problem in English** and get a working PINN?

That's what Lang-PINN does.

## Architecture Overview

Lang-PINN has **3 agents** and an **orchestrator**:

```
User: "Solve u'' + 2u' + 6400u = 0, u(0)=1, u'(0)=0"
                    |
              PDE Agent (LLM)
         Parse NL → structured PDESpec
                    |
             PINN Agent (rules or LLM)
         PDESpec → architecture recommendation
                    |
             Code Agent (template or LLM)
         PDESpec + arch → runnable Python code
                    |
              [Optional: Execute + Feedback loop]
```

The **Feedback Agent** (from `libs/pinn/feedback.py`) provides training quality scoring — it's the quality gate that tells the hybrid mode whether to accept or refine.

## The Three Modes

| Mode | PDE Agent | PINN Agent | Code Agent | Best for |
|------|-----------|------------|------------|----------|
| **library** | LLM | Rules | Template | Production, reproducibility |
| **code-agent** | LLM | LLM | LLM | Exploration, prototyping |
| **hybrid** | LLM | LLM | LLM + feedback | Best of both worlds |

- **Library mode**: Only the PDE parsing uses the LLM. Architecture and code generation use deterministic rules/templates derived from our 11 experiments. Fast, reproducible, always valid.
- **Code-agent mode**: All agents consult the LLM. More flexible but less predictable.
- **Hybrid mode**: LLM generates code, the `pinn` library executes it, the Feedback Agent scores it. If quality is below threshold, the LLM refines. A quality-gated loop.

## Hands-On: The PDESpec

The `PDESpec` is the contract between agents — a structured representation of the PDE problem. Let's build one manually first to understand the data flow.

In [ ]:
from lang_pinn import PDESpec

# Manually build a spec for the damped harmonic oscillator
spec = PDESpec(
    name="Damped Harmonic Oscillator",
    equation="u_tt + mu*u_t + k*u = 0",
    independent_vars=["t"],
    dependent_var="u",
    order=2,
    spatial_dim=0,
    domain={"t": (0.0, 1.0)},
    initial_conditions=["u(0) = 1", "u'(0) = 0"],
    parameters={"mu": 4.0, "k": 6400.0},
    has_high_frequency=True,
)

print(f"Name: {spec.name}")
print(f"Equation: {spec.equation}")
print(f"Order: {spec.order}, Spatial dim: {spec.spatial_dim}")
print(f"High frequency: {spec.has_high_frequency}")

## SymPy Verification

Before sending a spec downstream, we can verify it's mathematically consistent:

In [ ]:
from lang_pinn import verify_spec

# Valid spec — should pass
issues = verify_spec(spec)
print(f"Issues: {issues}")
assert len(issues) == 0, "Expected no issues"
print("All checks passed!")

In [ ]:
# Now let's try a deliberately broken spec
bad_spec = PDESpec(
    name="Broken",
    equation="u_tt + u_x = 0",  # claims order=1, but u_tt is order 2
    independent_vars=["t"],      # missing 'x' which appears in equation
    dependent_var="u",
    order=1,                      # wrong!
    spatial_dim=0,
    domain={"t": (0.0, 1.0)},
)

issues = verify_spec(bad_spec)
print("Detected issues:")
for issue in issues:
    print(f"  - {issue}")

## PINN Agent: Rule-Based Architecture

The PINN Agent recommends architecture based on PDE features. Let's see what it suggests for different problems:

In [ ]:
from lang_pinn import PINNAgent

agent = PINNAgent()

# ODE with high frequency
rec = agent.recommend(spec)
print(f"=== {spec.name} ===")
print(f"Network: {rec.hidden_layers}x{rec.hidden_neurons} {rec.activation}")
print(f"Ansatz: {rec.use_ansatz} ({rec.ansatz_type})")
print(f"Epochs: {rec.epochs}, LR: {rec.learning_rate}")
print(f"Collocation: {rec.n_collocation}")
print(f"Loss weights: {rec.loss_weights}")
print(f"Reasoning: {rec.reasoning}")

In [ ]:
# Compare: Burgers equation (1D PDE with sharp gradients)
burgers_spec = PDESpec(
    name="Burgers",
    equation="u_t + u*u_x = nu*u_xx",
    independent_vars=["x", "t"],
    dependent_var="u",
    order=2,
    spatial_dim=1,
    domain={"x": (-1.0, 1.0), "t": (0.0, 1.0)},
    parameters={"nu": 0.01},
    is_linear=False,
    has_sharp_gradients=True,
)

rec_burgers = agent.recommend(burgers_spec)
print(f"\n=== {burgers_spec.name} ===")
print(f"Network: {rec_burgers.hidden_layers}x{rec_burgers.hidden_neurons}")
print(f"Collocation: {rec_burgers.n_collocation} (more due to sharp gradients)")
print(f"Epochs: {rec_burgers.epochs}")
print(f"Reasoning: {rec_burgers.reasoning}")

Notice how the agent adapts:
- The oscillator gets a **sinusoidal Ansatz** and lower physics weight (1e-4)
- Burgers gets **5000 collocation points** (2.5x more due to sharp gradients) and more epochs

These heuristics come from our experience building 11 experiments.

## Code Agent: Template Generation

The Code Agent generates runnable Python code that uses the `pinn` library API:

In [ ]:
from lang_pinn import CodeAgent

code_agent = CodeAgent()
code = code_agent.generate(spec, rec, use_llm=False)  # template mode

print("Generated code (first 40 lines):")
print("=" * 60)
for i, line in enumerate(code.split("\n")[:40], 1):
    print(f"{i:3d} | {line}")

The generated code:
- Imports from `pinn` (our library)
- Creates a `PINN` model with the recommended architecture
- Sets up collocation points
- Defines loss functions with autograd derivatives
- Trains with `PINNTrainer` and `TrainingHealthMonitor`
- Evaluates quality

It's a valid Python script that you could save and run.

## The Full Pipeline: Orchestrator

The `Orchestrator` ties everything together. In library mode (no LLM needed for arch/code), we can run it end-to-end with just a spec:

In [ ]:
from lang_pinn import Orchestrator

# Library mode: PDE Agent uses LLM, but arch + code are deterministic
# Since we have a spec already, we can skip the PDE Agent entirely
orch = Orchestrator(mode="library")
result = orch.solve_from_spec(spec)

print(f"Mode: {result.mode}")
print(f"PDE: {result.spec.name}")
print(f"Architecture: {result.architecture.hidden_layers}x{result.architecture.hidden_neurons}")
print(f"Ansatz: {result.architecture.use_ansatz}")
print(f"Code lines: {result.code.count(chr(10)) + 1}")
print(f"Executed: {result.executed}")

## CLI Usage

From the terminal, you can use Lang-PINN without writing any Python:

```bash
# Full pipeline
uv run lang-pinn solve "u'' + 2u' + 6400u = 0, u(0)=1, u'(0)=0 on [0,1]"

# Parse only (see what the PDE Agent extracts)
uv run lang-pinn parse "heat equation on a rod"

# Parse + recommend (see architecture advice)
uv run lang-pinn recommend "Burgers equation with nu=0.01"

# Save generated code
uv run lang-pinn solve "exponential decay" --save-code -o my_experiment/
```

## Key Takeaways

1. **Lang-PINN separates concerns**: PDE parsing, architecture design, and code generation are independent agents
2. **Three modes** let you choose your trade-off: determinism vs flexibility
3. **Rule-based recommendations** encode heuristics from 11 real experiments
4. **SymPy verification** catches errors before they reach training
5. **The `pinn` library is the runtime** — Lang-PINN generates code that uses it, not raw PyTorch

**Next:** Notebook 11 dives into hybrid mode — the feedback loop in action.